In [1]:
import re
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

In [2]:
META_CSV = Path("../real_vase_files/master_metadata.csv")
PROFILES_PQ = Path(
    "../real_vase_files/all_profiles_120526.parquet"
)
OUT_DIR = Path(
    "pca_stratification_output/"
)
OUT_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
N_POINTS = 125
CAP = 20
FLOOR = 1
N_PCS_PLOT = 6
SIGMAS = [-2, -1, 0, 1, 2]
DATE_MIN = -1050
PERIOD_KEEP = re.compile(r"^(tang|song|northern_song|southern_song|liao|jurchen_jin|yuan|ming|qing)")
MISS_TOKENS = {"na", "no_data", ""}
SEED = 1

COL_GREEK = "#E69F00"
COL_CHINA = "#56B4E9"
COL_FILL  = "#C2185B"

In [4]:
print("Loading metadata...")
meta = pd.read_csv(META_CSV, dtype=str, keep_default_na=False, low_memory=False)
meta.head()

Loading metadata...


,gpp_no,collection,accession_number,culture_general,culture_specific,continent_made_found,country_made_found,region_made_found,date_earliest,date_latest,...,description,period,N_gpp_no,region_notes,original_filename,original_path,date_accessioned,source,notes,period_notes
0,nmk_5653,national_museum_korea,bongwan_12396,asian_east,chinese,asia,china,,-2000000,-10000,...,colorful earthenware. flat and stable.. colorf...,paleolithic,1,,nmk_5653.jpg,originals_staging/national_museum_korea/images...,01/05/2026,nmk_scrape_2026-04,na,na
1,nmk_91123,national_museum_korea,jeung_1404,asian_east,japanese,asia,japan,,-14000,-300,...,earthenware jar.,jomon,1,,nmk_91123.jpg,originals_staging/national_museum_korea/images...,01/05/2026,nmk_scrape_2026-04,na,na
2,met_52532,met_nyc,50.61.8,asian_east,chinese,asia,china,na,-10000,-2100,...,object_type: beaker,neolithic,1,na,na,na,na,na,na,
3,louvre_cl010135642,louvre_paris,ao 10927,asian_west,na,asia,iran,rayy,-8000,1244,...,vase piriform decoration: decoration linear no...,na,1,na,na,na,na,na,na,
4,nmk_2637,national_museum_korea,k_429,asian_east,korean,asia,korea,no_data,-8000,-1500,...,comb pattern earthenware. 3d view the comb-pat...,neolithic,1,na,nmk_2637.jpg,originals_staging/national_museum_korea/images...,01/05/2026,nmk_scrape_2026-04,na,na


In [5]:
gpp_profiles = pd.read_parquet(PROFILES_PQ)
NUM_POINTS = gpp_profiles["point_order"].value_counts()
print(f"Number of points per vase:\n{NUM_POINTS}")
gpp_profiles.head()

Number of points per vase:
point_order
0      107477
171    107477
158    107477
159    107477
160    107477
        ...  
88     107477
89     107477
90     107477
91     107477
249    107477
Name: count, Length: 250, dtype: int64


,gpp_no,x,y,point_order
0,amph_1002176,-0.012187,1.5,0
1,amph_1002176,0.025235,1.5,1
2,amph_1002176,0.062658,1.5,2
3,amph_1002176,0.100081,1.5,3
4,amph_1002176,0.137504,1.5,4


In [6]:
print("Loading parquet (gpp_no index)...")
gpp_with_profile = set(pq.read_table(PROFILES_PQ, columns=["gpp_no"])
                       .to_pandas()["gpp_no"].unique())
print(f"  pots with profile: {len(gpp_with_profile)}")

Loading parquet (gpp_no index)...
  pots with profile: 107477


In [7]:
# Greek subset
def to_num(x):
    return pd.to_numeric(x, errors="coerce")

meta["date_earliest_num"] = to_num(meta["date_earliest"])
meta["date_latest_num"]   = to_num(meta["date_latest"])

meta.head()

,gpp_no,collection,accession_number,culture_general,culture_specific,continent_made_found,country_made_found,region_made_found,date_earliest,date_latest,...,N_gpp_no,region_notes,original_filename,original_path,date_accessioned,source,notes,period_notes,date_earliest_num,date_latest_num
0,nmk_5653,national_museum_korea,bongwan_12396,asian_east,chinese,asia,china,,-2000000,-10000,...,1,,nmk_5653.jpg,originals_staging/national_museum_korea/images...,01/05/2026,nmk_scrape_2026-04,na,na,-2000000.0,-10000.0
1,nmk_91123,national_museum_korea,jeung_1404,asian_east,japanese,asia,japan,,-14000,-300,...,1,,nmk_91123.jpg,originals_staging/national_museum_korea/images...,01/05/2026,nmk_scrape_2026-04,na,na,-14000.0,-300.0
2,met_52532,met_nyc,50.61.8,asian_east,chinese,asia,china,na,-10000,-2100,...,1,na,na,na,na,na,na,,-10000.0,-2100.0
3,louvre_cl010135642,louvre_paris,ao 10927,asian_west,na,asia,iran,rayy,-8000,1244,...,1,na,na,na,na,na,na,,-8000.0,1244.0
4,nmk_2637,national_museum_korea,k_429,asian_east,korean,asia,korea,no_data,-8000,-1500,...,1,na,nmk_2637.jpg,originals_staging/national_museum_korea/images...,01/05/2026,nmk_scrape_2026-04,na,na,-8000.0,-1500.0


In [8]:
is_greek = meta["culture_specific"].isin(["greek", "greek_italiote"])
greek = meta[is_greek].copy()
greek = greek[(greek["date_earliest_num"].isna()) | (greek["date_earliest_num"] >= DATE_MIN)]
greek = greek[greek["gpp_no"].isin(gpp_with_profile)].copy()
greek["civ"] = "greek"
greek["dm"] = (greek["date_earliest_num"] + greek["date_latest_num"]) / 2
print(f"Greek/italiote in cohort: {len(greek)}")

Greek/italiote in cohort: 22792


In [9]:
# Chinese subset
is_chinese = meta["culture_specific"] == "chinese"
ch = meta[is_chinese].copy()
keep_period = ch["period"].isin(MISS_TOKENS) | ch["period"].str.contains(PERIOD_KEEP.pattern, na=False)
ch = ch[keep_period]
ch = ch[ch["gpp_no"].isin(gpp_with_profile)].copy()
ch["civ"] = "chinese"
ch["dm"] = np.nan
print(f"Chinese in cohort:        {len(ch)}")

Chinese in cohort:        43443


/var/folders/q1/fhsbr5cs3nv15vfnqhmblvpr0000gn/T/ipykernel_41658/1771693377.py:4: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  keep_period = ch["period"].isin(MISS_TOKENS) | ch["period"].str.contains(PERIOD_KEEP.pattern, na=False)


In [10]:
joint = pd.concat([greek, ch], ignore_index=True)
joint.head()

,gpp_no,collection,accession_number,culture_general,culture_specific,continent_made_found,country_made_found,region_made_found,date_earliest,date_latest,...,original_filename,original_path,date_accessioned,source,notes,period_notes,date_earliest_num,date_latest_num,civ,dm
0,amph_1010285,amphora_collection,"geneva, musee d'art et d'histoire: 16837",european,greek,europe,greece,attica_athens,-1050,-900,...,na,na,na,na,na,,-1050.0,-900.0,greek,-975.0
1,amph_1012067,amphora_collection,"warsaw, national museum: 138486",european,greek,europe,greece,attica_athens,-1050,-900,...,na,na,na,na,na,,-1050.0,-900.0,greek,-975.0
2,bapd_1000473,bapd,"cambridge, fitzwilliam museum: g115",european,greek,europe,greece,attica_athens,-1050,-900,...,na,na,na,na,na,,-1050.0,-900.0,greek,-975.0
3,bapd_1003042,bapd,"munich, antikensammlungen: 6157",european,greek,europe,greece,attica_athens,-1050,-900,...,bapd_1003042.jpg,global pot project master/images/originals/ba...,28/03/2026,pots_for_processing batch march 2026,na,,-1050.0,-900.0,greek,-975.0
4,bapd_1003604,bapd,"mainz, johannes gutenberg universitat: 52",european,greek,europe,greece,attica_athens,-1050,-900,...,pot_number1003604.0_url.img_3.jpg,global pot project master/images/originals/po...,30/03/2026,na,na,,-1050.0,-900.0,greek,-975.0


In [11]:
# Derive shape_class (Greek subdivisions; Chinese = shape_general)
ss = joint["shape_specific"].replace({
    "squat_lekythos": "squat",
    "neck-amphora":   "neck",
    "belly-amphora":  "belly",
})
sg = joint["shape_general"]

In [12]:
def shape_class_row(civ, sg, ss):
    if civ == "greek" and sg == "amphora":
        return {
            "neck":         "amphora_neck",
            "belly_type_b": "amphora_belly_b",
            "panathenaic":  "amphora_panathenaic",
            "belly_type_a": "amphora_belly_a",
            "belly":        "amphora_belly_unspec",
        }.get(ss, "amphora_other")
    if civ == "greek" and sg == "lekythos":
        return "lekythos_squat" if ss == "squat" else "lekythos_cylindrical"
    return sg

joint["shape_class"] = [shape_class_row(c, g, s) for c, g, s in zip(joint["civ"], sg, ss)]

In [13]:
# period_bin
def pb_row(row):
    if row["civ"] == "greek":
        return "unknown" if pd.isna(row["dm"]) else str(int(np.floor(row["dm"] / 50) * 50))
    if row["period"] in MISS_TOKENS:
        return "unknown"
    return row["period"]
joint["period_bin"] = joint.apply(pb_row, axis=1)

joint["region"] = joint["region_made_found"].where(
    ~joint["region_made_found"].isin(MISS_TOKENS), "unknown"
)
joint["cell"] = (joint["civ"] + "|" + joint["region"] + "|" +
                 joint["period_bin"] + "|" + joint["shape_class"])

print(f"Joint cohort:             {len(joint)}")
print(joint["civ"].value_counts())

Joint cohort:             66235
civ
chinese    43443
greek      22792
Name: count, dtype: int64


In [14]:
print("Loading profile points for joint cohort...")
prof = pq.read_table(PROFILES_PQ).to_pandas()
prof = prof[prof["gpp_no"].isin(set(joint["gpp_no"]))]

Loading profile points for joint cohort...


In [15]:
# Keep only pots with exactly N_POINTS
pt_counts = prof.groupby("gpp_no").size()
most_common = int(pt_counts.mode()[0])
if most_common != N_POINTS:
    print(f"Warning: N_POINTS={N_POINTS} but profiles have {most_common} points — updating N_POINTS.")
    N_POINTS = most_common
keep_ids = pt_counts[pt_counts == N_POINTS].index
prof = prof[prof["gpp_no"].isin(keep_ids)].sort_values(["gpp_no", "point_order"])
print(f"Pots with exactly {N_POINTS} points: {len(keep_ids)}")

Pots with exactly 250 points: 66235


In [16]:
# Pivot to wide: x_001..x_250 and y_001..y_250
print("Pivoting to wide matrix...")
xs = prof.pivot(index="gpp_no", columns="point_order", values="x")
ys = prof.pivot(index="gpp_no", columns="point_order", values="y")
gpp_index = xs.index.to_numpy()

Pivoting to wide matrix...


In [17]:
X = np.empty((len(gpp_index), 2 * N_POINTS), dtype=np.float64)
X[:, 0::2] = xs.values
X[:, 1::2] = ys.values
print(f"X shape: {X.shape}")

X shape: (66235, 500)


In [18]:
# Align joint meta to X row order
joint = joint.set_index("gpp_no").reindex(gpp_index).reset_index()
joint = joint.dropna(subset=["civ"])
mask  = np.isin(gpp_index, joint["gpp_no"].to_numpy())
X = X[mask]
gpp_index = gpp_index[mask]
joint = joint.set_index("gpp_no").loc[gpp_index].reset_index()
print(f"Aligned X shape: {X.shape}")

Aligned X shape: (66235, 500)


In [19]:
print("Cell-stratified subsample (cap=20)...")
rng = np.random.default_rng(SEED)
cell_groups = joint.groupby("cell").indices  # dict cell -> array of row idx
sel = []
for cell, idxs in cell_groups.items():
    if len(idxs) < FLOOR:
        continue
    take = min(len(idxs), CAP)
    sel.extend(rng.choice(idxs, size=take, replace=False))
sel = np.array(sorted(sel))
print(f"  subsample size: {len(sel)}")

X_sub  = X[sel]
civ_sub = joint["civ"].to_numpy()[sel]

Cell-stratified subsample (cap=20)...
  subsample size: 11913


In [20]:
print("PCA (stratified)...")
pca_strat = PCA(n_components=max(N_PCS_PLOT + 2, 10))
pca_strat.fit(X_sub)

PCA (stratified)...


PCA(n_components=10)

In [21]:
print("PCA (full)...")
pca_full = PCA(n_components=max(N_PCS_PLOT + 2, 10))
pca_full.fit(X)

PCA (full)...


PCA(n_components=10)

In [22]:
# Project FULL cohort onto each basis (for the scatter plots)
S_strat = pca_strat.transform(X)
S_full  = pca_full.transform(X)
civ_all = joint["civ"].to_numpy()

In [23]:
def scatter_pairs(S, ve, civ, title, out_path):
    pairs = [(0, 1), (2, 3), (4, 5)]
    fig, axes = plt.subplots(1, 3, figsize=(14, 5))
    for ax, (a, b) in zip(axes, pairs):
        for c, col in [("chinese", COL_CHINA), ("greek", COL_GREEK)]:
            sel = civ == c
            ax.scatter(S[sel, a], S[sel, b], s=2, alpha=0.25,
                       color=col, label=c, rasterized=True)
        ax.set_xlabel(f"PC{a+1} ({100*ve[a]:.1f}%)")
        ax.set_ylabel(f"PC{b+1} ({100*ve[b]:.1f}%)")
        ax.set_aspect("equal", adjustable="datalim")
    axes[-1].legend(loc="upper right", markerscale=4)
    fig.suptitle(title)
    fig.tight_layout()
    fig.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"  wrote: {out_path}")

scatter_pairs(S_strat, pca_strat.explained_variance_ratio_, civ_all,
              "Joint morphospace — cell-stratified PCA (cap=20)",
              OUT_DIR / "scatter_stratified.png")
scatter_pairs(S_full,  pca_full.explained_variance_ratio_,  civ_all,
              "Joint morphospace — full-cohort PCA",
              OUT_DIR / "scatter_full.png")

  wrote: pca_stratification_output/scatter_stratified.png
  wrote: pca_stratification_output/scatter_full.png


In [24]:
def backproject(mean, components, sds, pc_idx, sigma):
    """Mean shape + sigma SDs along PC. Returns (n_points, 2)."""
    score = sigma * sds[pc_idx]
    shape = mean + score * components[pc_idx]
    return shape.reshape(-1, 2)

def shape_modes_plot(pca, title, out_path, n_pcs=N_PCS_PLOT, sigmas=SIGMAS):
    sds = np.sqrt(pca.explained_variance_)
    mean = pca.mean_
    components = pca.components_

    # Precompute all shapes to set a common axis range
    all_shapes = []
    for pc in range(n_pcs):
        for s in sigmas:
            sh = backproject(mean, components, sds, pc, s)
            xs, ys = sh[:, 0], sh[:, 1]
            xs_full = np.concatenate([xs, -xs[::-1]])
            ys_full = np.concatenate([ys, ys[::-1]])
            all_shapes.append((xs_full, ys_full))

    all_x = np.concatenate([s[0] for s in all_shapes])
    all_y = np.concatenate([s[1] for s in all_shapes])
    pad_x = (all_x.max() - all_x.min()) * 0.05
    pad_y = (all_y.max() - all_y.min()) * 0.05
    xlim = (all_x.min() - pad_x, all_x.max() + pad_x)
    ylim = (all_y.min() - pad_y, all_y.max() + pad_y)

    fig, axes = plt.subplots(n_pcs, len(sigmas),
                             figsize=(len(sigmas) * 1.6, n_pcs * 1.8))
    if n_pcs == 1:
        axes = axes[np.newaxis, :]

    k = 0
    for pc in range(n_pcs):
        for j, s in enumerate(sigmas):
            ax = axes[pc, j]
            xs_full, ys_full = all_shapes[k]; k += 1
            ax.fill(xs_full, ys_full, color=COL_FILL, alpha=0.95,
                    edgecolor=COL_FILL, linewidth=0.5)
            ax.set_xlim(xlim); ax.set_ylim(ylim)
            ax.set_aspect("equal")
            ax.set_xticks([]); ax.set_yticks([])
            for spine in ax.spines.values():
                spine.set_visible(False)
            if pc == 0:
                ax.set_title(f"{s:+d}σ" if s != 0 else "mean",
                             fontsize=9)
            if j == 0:
                ax.set_ylabel(f"PC{pc+1}", rotation=0, ha="right",
                              va="center", fontsize=9)

    fig.suptitle(title, fontsize=11)
    fig.tight_layout(rect=[0, 0, 1, 0.97])
    fig.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"  wrote: {out_path}")

shape_modes_plot(pca_strat,
                 "Shape modes — cell-stratified PCA (cap=20)",
                 OUT_DIR / "shape_modes_stratified.png")
shape_modes_plot(pca_full,
                 "Shape modes — full-cohort PCA",
                 OUT_DIR / "shape_modes_full.png")

  wrote: pca_stratification_output/shape_modes_stratified.png
  wrote: pca_stratification_output/shape_modes_full.png


In [25]:
print()
print("Variance explained (top 10 PCs):")
print(f"{'PC':>4}  {'stratified':>10}  {'full':>10}")
for i in range(10):
    print(f"{i+1:>4}  {pca_strat.explained_variance_ratio_[i]:>10.4f}  "
          f"{pca_full.explained_variance_ratio_[i]:>10.4f}")
print(f"cum top-6 stratified: {pca_strat.explained_variance_ratio_[:6].sum():.4f}")
print(f"cum top-6 full:       {pca_full.explained_variance_ratio_[:6].sum():.4f}")


Variance explained (top 10 PCs):
  PC  stratified        full
   1      0.5697      0.7281
   2      0.2241      0.1576
   3      0.1043      0.0601
   4      0.0432      0.0217
   5      0.0244      0.0124
   6      0.0105      0.0068
   7      0.0056      0.0028
   8      0.0039      0.0025
   9      0.0029      0.0015
  10      0.0019      0.0010
cum top-6 stratified: 0.9762
cum top-6 full:       0.9867


In [26]:
print("Saving stratified profiles parquet...")
sel_gpp   = set(gpp_index[sel])
joint_gpp = set(joint["gpp_no"])

# -- stratified Greek/Chinese rows are already in `prof` (in memory) --
prof_strat = prof[prof["gpp_no"].isin(sel_gpp)]

# -- non-joint pots must come from disk, but read only what we need --
non_joint_gpp = list(gpp_with_profile - joint_gpp)
print(f"  non-joint pots to read: {len(non_joint_gpp)}")

import pyarrow as pa
import pyarrow.dataset as ds

dataset = ds.dataset(str(PROFILES_PQ), format="parquet")
prof_non_joint = dataset.to_table(
    filter=ds.field("gpp_no").isin(pa.array(non_joint_gpp, type=pa.string()))
).to_pandas()

prof_out = pd.concat([prof_strat, prof_non_joint], ignore_index=True)

out_path = "../real_vase_files/profiles_stratified_120526.parquet"
prof_out.to_parquet(out_path, index=False)
print(f"  pots saved: {prof_out['gpp_no'].nunique()}")
print(f"  rows saved: {len(prof_out)}")
print(f"  wrote: {out_path}")

Saving stratified profiles parquet...
  non-joint pots to read: 41242
  pots saved: 53155
  rows saved: 13288750
  wrote: ../real_vase_files/profiles_stratified_120526.parquet
